# 🛒 Capstone Project: Physical Infrastructure & Retail Layout Analytics 
> **Bridging Digital Transaction Sequences with Physical Retail Store Architecture**

---

## 📌 1. Executive Summary & Problem Statement

### 🔴 The Problem
Companies frequently design physical store layouts based on **intuition and guesswork** without knowing how spatial architectural choices (such as entrance heights, ceiling openness, aisle arrangements, and display positioning) directly affect customer movement, item pick sequence, and total basket sales. 

### 🟢 The Solution
Physical video tracking and in-store sensory networks are expensive and privacy-invasive. In this capstone project, we use e-commerce shopping cart sequence logs (`add_to_cart_order`) as a **data-driven proxy for physical customer store navigation**. 

By mapping item pick sequence numbers and department types into physical store zones, we evaluate how spatial store design choices influence customer purchasing behavior, item volume, and basket expansion.

---

## 🏛️ 2. Architectural Framework & Spatial Choices

Physical store layouts are strategically engineered around human environmental psychology: 

1. **Entrance Setups & Ceiling Heights:** 
    - *Architecture & Psychology:* High ceilings, open wide entryways, and vibrant fresh displays (*Bakery, Produce, Deli, Floral*) create a feeling of spaciousness and high quality. This lowers customer friction upon entry and triggers **sensory priming**. 
    - *Data Proxy:* Items selected during the first 3 pick positions (`add_to_cart_order <= 3`) in fresh departments are categorized under the **`Entrance_Sensory_Zone`**. 

2. **Aisle Layouts & Navigation Thoroughfares:** 
    - *Architecture & Psychology:* Core center-store aisles (*Pantry, Frozen, Canned Goods, Household*) manage steady traffic flow and guide shoppers toward essential staple goods. 
    - *Data Proxy:* Mid-trip pick orders (`3 < add_to_cart_order < 8`) and center-aisle departments are categorized under **`Main_Store_Aisles`**. 

3. **Checkout Queues & Register Real Estate:** 
    - *Architecture & Psychology:* Display racks placed directly along register lines capitalize on end-of-trip decision fatigue, capturing last-minute impulse purchases right before payment. 
    - *Data Proxy:* Grab-and-go items (*Snacks, Beverages*) picked near the end of the journey (`add_to_cart_order < 8`) are categorized under **`Checkout_Impulse_Zone`**.

---

## 🎯 3. Analytical Objectives & Core Business Questions 

This project answers three key retail infrastructure questions using DuckDB SQL analytics: 

1. **Physical Zone Baseline:** Does cart sequence data mathematically reflect a physical 3-zone journey from entrance (early pick) to checkout (late pick)? 
2. **The Entrance Priming Effect:** Do shoppers who visit fresh entrance setups first generate larger overall basket sizes than shoppers who bypass the entrance? 
3. **Department Sequence & Impulse Contribution:** Which departments act as high-volume "traffic anchors," and what proportion of total store sales originates from late-trip checkout impulse buys?

---

## 🛠️ 4. Data Stack & Methodology 
- **Data Ingestion & Feature Engineering:** Python (`pandas`) joins relational tables (`orders`, `products`, `departments`) and engineers spatial features. 
- **SQL Analytics Engine:** `DuckDB` executes high-speed spatial aggregations and CTEs on a 50,000-row transactional sample (`spatial_transactions_sample.csv`).

## Step 1: Data Ingestion & Schema Inspection 

I loaded three foundational CSV datasets from the Instacart Market Basket dataset to inspect their structures and key attributes:

- `order_products__prior.csv`: Captures shopping trip sequence details (`order_id`, `product_id`, `add_to_cart_order`).

- `products.csv`: Links specific product names to their department IDs.

- `departments.csv`: Defines store departments (e.g., *bakery*, *produce*, *snacks*, *frozen*).

In [ ]:
import pandas as pd

# 1. Loading the 3 CSV files
orders = pd.read_csv("Data/order_products__prior.csv")
products = pd.read_csv("Data/products.csv")
departments = pd.read_csv("Data/departments.csv")

# 2. Checking the first few rows of each table
print("=== ORDER PRODUCTS (Sequences) ===")
print(orders[["order_id", "product_id", "add_to_cart_order"]].head())

print('\n=== PRODUCTS ===')
print(products[["product_id", "product_name", "department_id"]].head())

print('\n=== DEPARTMENTS ===')
print(departments.head())

=== ORDER PRODUCTS (Sequences) ===
   order_id  product_id  add_to_cart_order
0         2       33120                  1
1         2       28985                  2
2         2        9327                  3
3         2       45918                  4
4         2       30035                  5

=== PRODUCTS ===
   product_id                                       product_name  \
0           1                         Chocolate Sandwich Cookies   
1           2                                   All-Seasons Salt   
2           3               Robust Golden Unsweetened Oolong Tea   
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...   
4           5                          Green Chile Anytime Sauce   

   department_id  
0             19  
1             13  
2              7  
3              1  
4             13  

=== DEPARTMENTS ===
   department_id department
0              1     frozen
1              2      other
2              3     bakery
3              4    produce
4    

### 💡 Key Takeaway — Data Verification

- **Data Integrity:** I successfully loaded and inspected the three primary relational tables (`orders`, `products`, and `departments`).

- **Sequential Tracking:** The `add_to_cart_order` field provides an explicit numerical sequence of when each item entered a customer's cart during their store trip, serving as our proxy for physical customer navigation.

### 🛍️ Retail Store Layout Design & Zone Definitions 

In physical grocery retail, store layouts are strategically designed around human psychology and shopper movement patterns. To translate digital transaction sequences into physical store movement, we categorize cart order (`add_to_cart_order`) into three core physical zones:

1. **`Entrance_Sensory_Zone`**
    - **Criteria:** Fresh, sensory departments (*Bakery, Produce, Deli, Floral*) added early in the trip (`add_to_cart_order <= 3`).
    - **Retail Psychology:** Stores place visual and aromatic fresh departments right at the front entrance to lower customer stress, boost mood, and establish a high-quality perception of the store immediately upon arrival.

2. **`Main_Store_Aisles`**
    - **Criteria:** Core middle-aisle categories (*Pantry, Frozen, Canned Goods, Household*) added mid-trip (`3 < add_to_cart_order < 8`).
    - **Retail Psychology:** This represents the main body of the store where shoppers spend the majority of their trip browsing standard grocery aisles and filling baseline household needs.

3. **`Checkout_Impulse_Zone`**
    - **Criteria:** Grab-and-go categories (*Snacks, Beverages*) added late in the trip (`add_to_cart_order >= 8`).
    - **Retail Psychology:** Placed near register queues to capitalize on decision fatigue at the end of a visit, encouraging high-margin, last-minute additions right before payment.

## Step 2: Spatial Feature Engineering & Zone Mapping

I merged my transactional tables together and engineered a new spatial feature, `store_zone`, based on item cart sequence (`add_to_cart_order`) and department type:

- **`Entrance_Sensory_Zone`**: High-sensory fresh departments (*bakery*, *produce*, *deli*, *floral*) selected early in the trip (items 1–3). 

- **`Checkout_Impulse_Zone`**: Impulse categories (*snacks*, *beverages*) selected late in the trip (items 8+). 

- **`Main_Store_Aisles`**: Core mid-trip store navigation across all remaining categories.  (items 4 to 7)

Finally, I exported a processed sample to `Data/spatial_transactions_sample.csv` for local SQL analytics.

In [7]:
import pandas as pd

# 1. Loading the CSV files
orders = pd.read_csv("Data/order_products__prior.csv")
products = pd.read_csv("Data/products.csv")
departments = pd.read_csv("Data/departments.csv")

# 2. Merging the tables together
df = orders.merge(products, on="product_id", how="left")
df = df.merge(departments, on="department_id", how="left")

# 3. Defining my Physical Store Zone mapping function
def assign_store_zone(row):
    dept = str(row['department']).lower()
    seq = row["add_to_cart_order"]

    if dept in ["bakery", "produce", "deli", "floral"] and seq <= 3:
        return "Entrance_Sensory_Zone"
    elif dept in ["snacks", "beverages"] and seq >= 8:
        return "Checkout_Impulse_Zone"
    else:
        return "Main_Store_Aisles"

# 4. Applying the mapping to create a new 'store_zone' column
df["store_zone"] = df.apply(assign_store_zone, axis=1)

# 5. Inspecting a sample of my mapped data
print("=== MAPPED SPATIAL TRANSACTIONS ===")
print(df[["order_id", "product_name", "department", "add_to_cart_order", "store_zone"]].head(10))

# 6. Saving a processed sample CSV for my BigQuery step
df.head(50000).to_csv("Data/spatial_transactions_sample.csv", index=False)
print("\nSuccess! Saved 'Data/spatial_transactions_sample.csv'")

=== MAPPED SPATIAL TRANSACTIONS ===
   order_id                                       product_name  department  \
0         2                                 Organic Egg Whites  dairy eggs   
1         2                              Michigan Organic Kale     produce   
2         2                                      Garlic Powder      pantry   
3         2                                     Coconut Butter      pantry   
4         2                                  Natural Sweetener      pantry   
5         2                                            Carrots     produce   
6         2                   Original Unflavored Gelatine Mix      pantry   
7         2           All Natural No Stir Creamy Almond Butter      pantry   
8         2                            Classic Blend Cole Slaw     produce   
9         3  Total 2% with Strawberry Lowfat Greek Strained...  dairy eggs   

   add_to_cart_order             store_zone  
0                  1      Main_Store_Aisles  
1            

### 💡 Key Takeaway — Spatial Zone Creation

- **Spatial Mapping:** We transformed raw transactional sequence records into three distinct physical retail zones: `Entrance_Sensory_Zone`, `Main_Store_Aisles`, and `Checkout_Impulse_Zone`.

- **Sample Export:** We successfully compiled and exported a 50,000-row sample (`spatial_transactions_sample.csv`) for local high-speed SQL analytics in DuckDB.

## Step 3: Baseline SQL Analysis — Physical Zone Summary 

I used DuckDB to execute a local SQL query against `Data/spatial_transactions_sample.csv`. This aggregates total item counts and calculates the average item pick sequence (`avg_item_pick_order`) for each physical store zone to verify that my spatial mapping accurately distinguishes early entrance picks from late impulse buys.

In [11]:
import duckdb

# Running SQL directly on my CSV file with clear column names
query = """
SELECT
    store_zone AS physical_store_zone,
    COUNT(order_id) AS total_items_purchased,
    ROUND(AVG(add_to_cart_order), 2) AS avg_item_pick_order
FROM 'Data/spatial_transactions_sample.csv'
GROUP BY store_zone;
"""

duckdb.sql(query).show()

┌───────────────────────┬───────────────────────┬─────────────────────┐
│  physical_store_zone  │ total_items_purchased │ avg_item_pick_order │
│        varchar        │         int64         │       double        │
├───────────────────────┼───────────────────────┼─────────────────────┤
│ Main_Store_Aisles     │                 41402 │                8.39 │
│ Entrance_Sensory_Zone │                  5106 │                 2.0 │
│ Checkout_Impulse_Zone │                  3492 │               14.39 │
└───────────────────────┴───────────────────────┴─────────────────────┘



### 💡 Key Takeaway — Physical Zone Behaviors

- **Entrance Validation (`avg_item_pick_order = 2.00`):** Fresh, sensory items (bakery, produce, deli) are picked up almost immediately at the start of the trip (averaging item #2). 
- **Mid-Trip Bulk (`avg_item_pick_order = 8.39`):** Over 82% of total cart volume (41,402 items) is gathered mid-trip across the main store aisles.

- **Checkout Timing (`avg_item_pick_order = 14.39`):** Impulse purchases occur nearly 7x later in sequence position than entrance picks, confirming a distinct end-of-trip purchasing phase before payment.

## Step 4: Question 1 — Testing the Entrance Priming Effect

I tested whether visiting high-sensory fresh departments (bakery, produce, deli) at the start of a shopping trip correlates with larger overall basket sizes.

We use a SQL Common Table Expression (CTE) to categorize each order into two shopper groups:

1. **Visited Entrance Zone First:** Orders containing an entrance sensory item. 
2. **Skipped Entrance Zone:** Orders that bypassed the entrance and went directly to main aisles.

In [ ]:
import duckdb
# Query 2: Entrance Priming Effect Analysis
query_entrance = """
WITH OrderSummary AS (
     SELECT
        order_id,
        COUNT(*) AS total_basket_items,
        MAX(CASE WHEN store_zone = 'Entrance_Sensory_Zone' THEN 1 ELSE 0 END) AS visited_entrance_first
        FROM 'Data/spatial_transactions_sample.csv'
        GROUP BY order_id
)

SELECT
    CASE
        WHEN visited_entrance_first = 1 THEN 'Visited Entrance Zone First'
        ELSE 'Skipped Entrance Zone'
        END AS shopper_group,
        COUNT(order_id) AS total_orders,
        ROUND(AVG(total_basket_items), 2) AS avg_basket_size
    FROM OrderSummary
    GROUP BY visited_entrance_first
    ORDER BY avg_basket_size DESC;
"""

duckdb.sql(query_entrance).show()

┌─────────────────────────────┬──────────────┬─────────────────┐
│        shopper_group        │ total_orders │ avg_basket_size │
│           varchar           │    int64     │     double      │
├─────────────────────────────┼──────────────┼─────────────────┤
│ Visited Entrance Zone First │         3142 │           10.73 │
│ Skipped Entrance Zone       │         1836 │            8.87 │
└─────────────────────────────┴──────────────┴─────────────────┘



### 💡 Key Takeaway — Entrance Priming Impact 

- **Basket Expansion:** Shoppers who begin their trip by selecting fresh/sensory entrance items generate significantly larger overall basket sizes compared to shoppers who bypass the entrance. 

- **Layout Strategy:** This confirms the psychological impact of entrance priming—starting the visit with visual and aromatic fresh items lowers shopper friction and encourages broader store browsing.

## Step 5: Question 4 — Department Popularity (Anchors vs. Low-Volume Zones)

I aggregated total item purchases by department to identify high-volume "anchor departments" (that draw primary traffic) versus lower-volume departments. 

I also included average pick sequence (`avg_item_pick_order`) to understand when during the trip each category is selected.

In [ ]:
# Query 3: Department Volume & Pick Order Ranking
query_departments = """
SELECT
    department AS department_name,
    COUNT(*) AS total_items_purchased,
    ROUND(AVG(add_to_cart_order), 2) AS avg_item_pick_order
FROM 'Data/spatial_transactions_sample.csv'
GROUP BY department
ORDER BY total_items_purchased DESC;
"""

duckdb.sql(query_departments).show()

┌─────────────────┬───────────────────────┬─────────────────────┐
│ department_name │ total_items_purchased │ avg_item_pick_order │
│     varchar     │         int64         │       double        │
├─────────────────┼───────────────────────┼─────────────────────┤
│ produce         │                 14542 │                7.86 │
│ dairy eggs      │                  8443 │                7.47 │
│ snacks          │                  4456 │                8.94 │
│ beverages       │                  4139 │                 6.9 │
│ frozen          │                  3398 │                8.47 │
│ pantry          │                  2899 │                9.16 │
│ bakery          │                  1878 │                7.85 │
│ canned goods    │                  1669 │                9.43 │
│ deli            │                  1572 │                 8.7 │
│ dry goods pasta │                  1286 │                9.87 │
│ household       │                  1124 │                7.89 │
│ breakfas

### 💡 Key Takeaway — Department Volume & Pick Order Ranking 

- **Primary Traffic Anchors:** **Produce** (14,542 items) and **Dairy & Eggs** (8,443 items) dominate overall volume, accounting for nearly half of all item purchases and serving as essential destination anchors for store foot traffic.

- **Trip Sequence & Timing Insights:**

    - **Alcohol (`5.38`)** is selected earliest in the trip, indicating deliberate, planned purchases upon store entry.

    - **Beverages (`6.90`)** are picked up significantly earlier than **Snacks (`8.94`)**, showing that liquid refreshments are gathered mid-way through main aisle navigation, while dry snacks are held back closer to checkout.

    - **Dry Goods & Pasta (`9.87`)** and **Babies (`11.00`)** are selected near the tail end of the trip, making them prime candidates for cross-merchandising along secondary walkways.

## Step 6: Question 2 — Analyzing Checkout Impulse Contribution 

We measure the proportion of total store volume originating from late-trip impulse purchases in the `Checkout_Impulse_Zone`. We evaluate how significantly end-of-trip snack and beverage picks contribute to overall basket volume.

In [16]:
import duckdb
# Query 4: Checkout Impulse Volume & Percentage Share
query_impulse = """
SELECT
    store_zone,
    COUNT(*) AS total_items_purchased,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM 'Data/spatial_transactions_sample.csv'), 2) AS pct_total_volume
    FROM 'Data/spatial_transactions_sample.csv'
    GROUP BY store_zone
    ORDER BY total_items_purchased DESC;
    """ 

duckdb.sql(query_impulse).show()

┌───────────────────────┬───────────────────────┬──────────────────┐
│      store_zone       │ total_items_purchased │ pct_total_volume │
│        varchar        │         int64         │      double      │
├───────────────────────┼───────────────────────┼──────────────────┤
│ Main_Store_Aisles     │                 41402 │             82.8 │
│ Entrance_Sensory_Zone │                  5106 │            10.21 │
│ Checkout_Impulse_Zone │                  3492 │             6.98 │
└───────────────────────┴───────────────────────┴──────────────────┘



### 💡 Key Takeaway — Checkout Impulse Volume & Contribution 

- **Impulse Share (6.98%):** Late-trip impulse items account for nearly **7% of total store purchase volume** (3,492 items). While 7% represents a modest share of total physical volume, these checkout-adjacent items (snacks, single-serve beverages) typically carry significantly higher profit margins than center-store staples.

- **Main Aisle Navigation (82.80%):** Over **82% of total purchases** take place within the main store aisles, confirming that middle-aisle traffic flow is the primary revenue driver for overall basket fill.

- **Retail Strategy:** Securing nearly 7% of store volume right before register queuing highlights the importance of register end-cap design. Optimizing front-end display layouts directly captures high-margin revenue from decision-fatigued shoppers before payment.